# Sistema de Predicción Temprana de Plagas - Sierra del Patlachique
Este sistema analiza variables climáticas y edáficas para identificar condiciones de riesgo en cultivos. La zona de estudio se encuentra en la Sierra del Patlachique, Estado de México.

### Variables Estáticas (Contexto):
* **Altitud:** ~2,500 msnm (Influye en la oscilación térmica).
* **Tipo de Suelo:** Predominio de Phaeozems y Vertisoles (Suelos arcillosos con alta retención de humedad).

In [15]:
import pandas as pd
import requests
from datetime import datetime

# Configuración de ubicación y fechas
LAT, LON = 19.65, -98.85
FECHA_INICIO = "20240101"
FECHA_FIN = datetime.now().strftime("%Y%m%d")

# Consulta a la API de la NASA POWER
params = "T2M,PRECTOTCORR,GWETTOP,RH2M"
url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters={params}&community=AG&longitude={LON}&latitude={LAT}&start={FECHA_INICIO}&end={FECHA_FIN}&format=JSON"

print("Extracción de datos en proceso...")
res = requests.get(url)
raw_data = res.json()['properties']['parameter']

# Crear DataFrame y renombrar columnas para mayor claridad
df_patlachique = pd.DataFrame(raw_data)
df_patlachique.index = pd.to_datetime(df_patlachique.index)

# Diccionario de variables:
# T2M -> temp_media (°C)
# PRECTOTCORR -> lluvia_mm (Precipitación diaria)
# GWETTOP -> humedad_suelo (Escala 0-1, saturación de la capa superficial)
# RH2M -> humedad_relativa (%)
df_patlachique.columns = ['temp_media', 'lluvia_mm', 'humedad_suelo', 'humedad_relativa']
df_patlachique.index.name = 'fecha'

print("Datos cargados.")
df_patlachique.head()

Extracción de datos en proceso...
Datos cargados.


,temp_media,lluvia_mm,humedad_suelo,humedad_relativa
fecha,,,,
2024-01-01,10.99,0.00,0.45,55.47
2024-01-02,10.87,0.00,0.45,58.37
2024-01-03,11.45,0.19,0.44,52.55
2024-01-04,10.84,0.04,0.44,70.67
2024-01-05,10.86,0.01,0.44,68.84


### Significado de las Variables de Predicción:
1. **Temperatura Media (temp_media):** Es el motor biológico. En altitudes de 2,500 msnm, temperaturas constantes por encima de los 18°C aceleran drásticamente el ciclo de reproducción de los pulgones. Un aumento térmico tras un periodo frío es la señal de alerta para la eclosión de huevos de chapulín.
2. **Lluvia (lluvia_mm):** Actúa como un regulador físico. Lluvias intensas pueden "lavar" mecánicamente a los pulgones de las hojas, reduciendo la plaga. Sin embargo, la ausencia de lluvia (sequía) debilita las defensas naturales de la alfalfa y el maíz, haciéndolos más vulnerables a insectos chupadores.
3. **Humedad del Suelo (humedad_suelo):** Variable edáfica crítica para el chapulín. Dado que los suelos del Patlachique son arcillosos (Vertisoles), retienen la humedad de forma prolongada. Un pico de humedad en el suelo seguido de semanas secas permite que los huevecillos de chapulín enterrados sobrevivan y nazcan masivamente al subir la temperatura.
4. **Humedad Relativa (humedad_relativa):** A diferencia de otras regiones, aquí la baja humedad relativa (ambientes secos) es la que dispara el riesgo de ácaros y pulgones. Cuando el aire es seco, estos insectos succionan más savia para mantenerse hidratados, causando un daño doble al cultivo por deshidratación y extracción de nutrientes.

### Generación de Indicadores de Riesgo

In [16]:
# 1. Riesgo de Chapulín (Masticador - Maíz/Alfalfa)
# Eclosión tras humedad en suelo hace 15 días y calor actual
df_patlachique['riesgo_chapulin'] = (df_patlachique['humedad_suelo'].shift(15) > 0.45) & (df_patlachique['temp_media'] > 18)

# 2. Riesgo de Pulgón (Chupador - Alfalfa)
# Prefiere calor, falta de lluvia y ambiente seco
df_patlachique['riesgo_pulgon'] = (df_patlachique['temp_media'] > 20) & (df_patlachique['lluvia_mm'] == 0) & (df_patlachique['humedad_relativa'] < 60)

# 3. Riesgo de Gusano Cogollero (Barrenador - Maíz)
# Requiere acumulación de calor (>12°C) y humedad acumulada en la última semana
# para que las pupas emerjan del suelo
df_patlachique['riesgo_cogollero'] = (df_patlachique['temp_media'] > 12) & (df_patlachique['lluvia_mm'].rolling(window=7).sum() > 5)

# 4. Riesgo de Araña Roja (Ácaro - Alfalfa/Frijol)
# Indicador de sequía extrema: calor alto (>25°C) y aire muy seco (<45%)
df_patlachique['riesgo_arana_roja'] = (df_patlachique['temp_media'] > 25) & (df_patlachique['humedad_relativa'] < 45)

# 5. Estrés Hídrico (Vulnerabilidad de la Flora)
df_patlachique['estres_hidrico'] = df_patlachique['humedad_suelo'] < 0.25

# --- Semáforo de Alerta General ---
# Sumamos todos los riesgos de plagas activos
plagas = ['riesgo_chapulin', 'riesgo_pulgon', 'riesgo_cogollero', 'riesgo_arana_roja']
df_patlachique['nivel_alerta'] = df_patlachique[plagas].sum(axis=1)

print("Indicadores biológicos calculados.")
print(f"Días totales con algún nivel de riesgo: {len(df_patlachique[df_patlachique['nivel_alerta'] > 0])}")

Indicadores biológicos calculados.
Días totales con algún nivel de riesgo: 396


### Almacenamiento y Persitencia

In [17]:
# Definir la ruta de guardado
ruta_archivo = "../data/raw/datos_patlachique_procesados.csv"

# Guardar el archivo
# index=True guarda las fechas
df_patlachique.to_csv(ruta_archivo, index=True, encoding='utf-8')

print(f"Información guardada exitosamente en: {ruta_archivo}")

Información guardada exitosamente en: ../data/raw/datos_patlachique_procesados.csv
